# Merge EPA Water Quality/Climate dataset with Agriculture

In [1]:
import pandas as pd
import numpy as np

# Load EPA-Climate merged data
epa_climate_df = pd.read_csv('../../data/tabular/merged/epa-climate-merged.csv')

# Load Agriculture data
ag_df = pd.read_csv('../../data/tabular/agricultural/clean/usdaNass-agriculture-clean.csv')

In [6]:
epa_climate_df.columns

Index(['MonitoringLocationIdentifier', 'ActivityStartDateTime',
       'Ammonia-nitrogen_value', 'Ammonia-nitrogen_unit', 'Chloride_value',
       'Chloride_unit', 'Chlorophyll a, free of pheophytin_value',
       'Chlorophyll a, free of pheophytin_unit', 'Copper_value', 'Copper_unit',
       'Dissolved oxygen (DO)_value', 'Dissolved oxygen (DO)_unit',
       'Escherichia coli_value', 'Escherichia coli_unit',
       'Kjeldahl nitrogen_value', 'Kjeldahl nitrogen_unit',
       'Microcystin_value', 'Microcystin_unit', 'Nitrate_value',
       'Nitrate_unit', 'Nitrate + Nitrite_value', 'Nitrate + Nitrite_unit',
       'Nitrite_value', 'Nitrite_unit', 'Orthophosphate_value',
       'Orthophosphate_unit',
       'Precipitation 24hr prior to monitoring event amount_value',
       'Precipitation 24hr prior to monitoring event amount_unit',
       'Specific conductance_value', 'Specific conductance_unit',
       'Sulfate_value', 'Sulfate_unit', 'Temperature, air_value',
       'Temperature, air_

In [3]:
epa_climate_df.head()

,MonitoringLocationIdentifier,ActivityStartDateTime,Ammonia-nitrogen_value,Ammonia-nitrogen_unit,Chloride_value,Chloride_unit,"Chlorophyll a, free of pheophytin_value","Chlorophyll a, free of pheophytin_unit",Copper_value,Copper_unit,...,distance_to_climate_station_km,doy,gdd_40_86,high,highc,low,lowc,precip,snow,snowd
0,21IOWA_WQX-10030001,2024-01-02 10:10:00,NaN,NaN,16.0,mg/L,3.0,ug/L,NaN,NaN,...,17.149934,2,0.0,23.0,-5.0,18.0,-7.8,0.00,0.0,0.0
1,21IOWA_WQX-10030001,2024-02-01 11:10:00,NaN,NaN,14.0,mg/L,8.0,ug/L,NaN,NaN,...,17.149934,32,2.5,45.0,7.2,28.0,-2.2,0.00,0.0,0.0
2,21IOWA_WQX-10030001,2024-03-12 10:00:00,NaN,NaN,16.0,mg/L,9.0,ug/L,NaN,NaN,...,17.149934,72,13.5,67.0,19.4,39.0,3.9,0.00,0.0,0.0
3,21IOWA_WQX-10030001,2024-04-08 09:55:00,NaN,NaN,20.0,mg/L,34.0,ug/L,NaN,NaN,...,17.149934,99,2.5,45.0,7.2,39.0,3.9,0.25,0.0,0.0
4,21IOWA_WQX-10030001,2024-05-07 11:15:00,NaN,NaN,25.0,mg/L,30.0,ug/L,1.6,ug/L,...,17.149934,128,21.5,70.0,21.1,53.0,11.7,0.74,0.0,0.0


In [7]:
print("Preparing data for merge...")
# Ensure date columns are datetime
epa_climate_df['day_dt'] = pd.to_datetime(epa_climate_df['ActivityStartDateTime'], errors='coerce')
epa_climate_df['year'] = epa_climate_df['day_dt'].dt.year
epa_climate_df['state'] = 'IOWA' 

print("Pivoting agriculture data to wide format...")
# Clean up Data Item names for column headers
ag_df['data_item_short'] = ag_df['data_item'].str.replace(', MEASURED IN .*', '', regex=True)

# Pivot
ag_pivot = ag_df.pivot_table(
    index=['year', 'period', 'state'],
    columns='data_item_short',
    values='value',
    aggfunc='first'
).reset_index()

# Create month mapping for Ag data
months = ['JAN', 'FEB', 'MAR', 'APR', 'MAY', 'JUN', 'JUL', 'AUG', 'SEP', 'OCT', 'NOV', 'DEC']
ag_pivot['month'] = ag_pivot['period'].where(ag_pivot['period'].isin(months))

# For dates in epa_climate, get the month
epa_climate_df['month'] = epa_climate_df['day_dt'].dt.strftime('%b').str.upper()

Preparing data for merge...
Pivoting agriculture data to wide format...


In [8]:
print("Merging on Year, Month, and State...")
merged_df = epa_climate_df.merge(
    ag_pivot[ag_pivot['month'].notna()],
    left_on=['year', 'month', 'state'],
    right_on=['year', 'month', 'state'],
    how='left'
)

print(f"Final merged shape: {merged_df.shape}")

Merging on Year, Month, and State...
Final merged shape: (6581, 3884)


In [9]:
merged_df.head(5)

,MonitoringLocationIdentifier,ActivityStartDateTime,Ammonia-nitrogen_value,Ammonia-nitrogen_unit,Chloride_value,Chloride_unit,"Chlorophyll a, free of pheophytin_value","Chlorophyll a, free of pheophytin_unit",Copper_value,Copper_unit,...,"WOODY ORNAMENTALS & VINES, OTHER, RETAIL - OPERATIONS WITH SALES","WOODY ORNAMENTALS & VINES, OTHER, VINCA - OPERATIONS WITH INVENTORY","WOODY ORNAMENTALS & VINES, OTHER, VINCA - OPERATIONS WITH SALES","WOODY ORNAMENTALS & VINES, OTHER, VINCA - SALES","WOODY ORNAMENTALS & VINES, OTHER, VINCA, RETAIL - OPERATIONS WITH SALES","WOODY ORNAMENTALS & VINES, OTHER, VINCA, WHOLESALE - OPERATIONS WITH SALES","WOODY ORNAMENTALS & VINES, OTHER, WHOLESALE - OPERATIONS WITH SALES",WOOL - PRICE RECEIVED,WOOL - PRODUCTION,WOOL - SHORN
0,21IOWA_WQX-10030001,2024-01-02 10:10:00,NaN,NaN,16.0,mg/L,3.0,ug/L,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,21IOWA_WQX-10030001,2024-02-01 11:10:00,NaN,NaN,14.0,mg/L,8.0,ug/L,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,21IOWA_WQX-10030001,2024-03-12 10:00:00,NaN,NaN,16.0,mg/L,9.0,ug/L,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,21IOWA_WQX-10030001,2024-04-08 09:55:00,NaN,NaN,20.0,mg/L,34.0,ug/L,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,21IOWA_WQX-10030001,2024-05-07 11:15:00,NaN,NaN,25.0,mg/L,30.0,ug/L,1.6,ug/L,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [10]:
# Save the result (this may take a while due to the size of the data)
output_path = '../../data/tabular/merged/final-data.csv'
merged_df.to_csv(output_path, index=False)
print(f"Saved merged data to {output_path}")

Saved merged data to ../../data/tabular/merged/final-data.csv
